<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/03_advanced/03_art_agentic_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08. Agentic RL：ART フル学習 + Weave トレース + W&B Models

OpenPipe ART で小型モデルをツール利用エージェントとして強化学習し、
W&B Weave で軌跡をトレース、W&B Models に学習済み LoRA を保存します。

## 学習内容
1. ツール利用タスクの設計と報酬関数
2. ART トレーニングループ（ローカル GPU）
3. 各イテレーションの軌跡を Weave でトレース
4. W&B でトレーニングメトリクスを可視化
5. 学習済み LoRA を W&B Artifact として保存

> **GPU 必須（T4 以上）**: Colab → ランタイム → GPU を変更 → T4


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader


In [ ]:
# ART のインストール（Python >=3.11 / GPU 必須）
# Colab: ランタイム → GPU を変更 → T4 を選択してから実行
!pip install -q uv
!uv pip install --system -q \
  "openpipe-art[backend]" \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "python-dotenv>=1.0.0"

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv; load_dotenv()
    # ローカル: .env に空欄で書かれていた場合も削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")


In [ ]:
import art
import wandb
import weave
import asyncio
import re
import json
from art.local import LocalBackend
from art import dev
from pathlib import Path
import torch

# V100 / Volta (CC 7.0) 互換パッチ
import vllm.platforms.cuda as _cuda_platform
@classmethod
def _v100_punica(cls): return "vllm.lora.punica_wrapper.punica_cpu.PunicaWrapperCPU"
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8:
    _cuda_platform.CudaPlatform.get_punica_wrapper = _v100_punica

# CC < 8.0 (V100) 用の設定
_IS_OLD_GPU = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8
_V100_CONFIG = dev.InternalModelConfig(
    engine_args=dev.EngineArgs(
        enable_sleep_mode=False,
        enforce_eager=True,
    ),
    init_args=dev.InitArgs(
        gpu_memory_utilization=0.7,
        max_seq_length=4096,
    ),
) if _IS_OLD_GPU else None

wandb.login()
weave_client = weave.init(f"{ENTITY}/weave-course-art")
print("初期化完了")

## 1. タスク設計：ツール利用算数エージェント

エージェントは `<tool>` タグで計算ツールを呼び出せます。
複数のステップが必要な問題を解くことで、
**ツールを正しく使う能力**を強化学習で習得させます。

```
System: You have a calculator tool. To use it, write: <tool>{"op": "calc", "expr": "..."}</tool>
User:   Apples cost 3 each. Bob buys 7. What is the total?
Agent:  <tool>{"op": "calc", "expr": "3 * 7"}</tool>
Tool:   21
Agent:  The total is 21.
```


In [ ]:
import math as _math

SYSTEM_PROMPT = """You are a math assistant with a calculator tool.
To calculate, write exactly: <tool>{"op": "calc", "expr": "EXPRESSION"}</tool>
The result will be returned and you can use it to answer.
Always use the calculator for arithmetic. Final answer: write only the number."""

# 問題セット（難易度別）
PROBLEMS = [
    # (question, answer)
    ("What is 6 times 7?",                                    42),
    ("What is 144 divided by 12?",                            12),
    ("A store sells pens for 8 each. How much for 9 pens?",   72),
    ("What is 15 squared?",                                   225),
    ("What is the square root of 256?",                       16),
    ("What is 3 to the power of 5?",                          243),
    ("Bob has 50 coins. He spends 17. How many remain?",       33),
    ("What is 7 times 8 plus 6?",                             62),
]

def execute_tool(tool_json_str: str) -> str:
    """<tool>タグの中身を実行して結果を返す"""
    try:
        tool_call = json.loads(tool_json_str)
        if tool_call.get("op") == "calc":
            expr = tool_call["expr"]
            result = eval(expr, {"__builtins__": {}}, {"math": _math, "sqrt": _math.sqrt})
            return str(result)
        return "Unknown tool"
    except Exception as e:
        return f"Error: {e}"

def extract_reward(final_output: str, expected_answer: int) -> float:
    """最終出力から数値を抽出して正解と比較"""
    numbers = re.findall(r"-?\d+(?:\.\d+)?", final_output.replace(",", ""))
    if not numbers:
        return 0.0
    predicted = float(numbers[-1])
    return 1.0 if abs(predicted - expected_answer) < 0.5 else 0.0

print("タスク定義完了:", len(PROBLEMS), "問")


## 2. ロールアウト関数

In [ ]:
@weave.op(name="art_rollout")
async def rollout(
    model: art.TrainableModel,
    question: str,
    expected_answer: int,
    max_turns: int = 4,
) -> dict:
    """
    1 エピソードを実行して Trajectory と結果を返す。
    @weave.op でトレースされるためWeave UIで軌跡を確認できる。
    """
    traj = art.Trajectory(
        messages_and_choices=[{"role": "system", "content": SYSTEM_PROMPT}],
        reward=0.0,
        metrics={"expected": expected_answer},
    )
    traj.messages_and_choices.append({"role": "user", "content": question})

    client = model.openai_client()
    final_output = ""

    for turn in range(max_turns):
        resp = await client.chat.completions.create(
            model=model.inference_model_name,
            messages=traj.messages(),
            max_completion_tokens=128,
            stop=["</tool>"],
        )
        choice = resp.choices[0]
        output = (choice.message.content or "") + (
            "</tool>" if choice.finish_reason == "stop" and "<tool>" in (choice.message.content or "") else ""
        )

        # <tool> タグを検出して実行
        tool_match = re.search(r"<tool>(.*?)</tool>", output, re.DOTALL)
        if tool_match:
            tool_result = execute_tool(tool_match.group(1))
            traj.messages_and_choices.append({"role": "assistant", "content": output})
            traj.messages_and_choices.append({"role": "user", "content": f"Tool result: {tool_result}"})
            final_output = tool_result
        else:
            traj.messages_and_choices.append(choice)
            final_output = output
            break  # ツール呼び出しなし → 最終回答

    reward = extract_reward(final_output, expected_answer)
    traj.reward = reward
    traj.metrics["predicted_output"] = final_output
    traj.metrics["reward"] = reward

    return {
        "trajectory": traj,
        "question": question,
        "expected": expected_answer,
        "output": final_output,
        "reward": reward,
    }

print("ロールアウト関数定義完了")


## 3. トレーニングループ

In [ ]:
# ハイパーパラメータ
NUM_ITERATIONS  = 20   # トレーニングイテレーション数（Colab T4 で約 30-60 分）
ROLLOUTS_PER_IT = 8    # 各イテレーションのロールアウト数
LEARNING_RATE   = 5e-5

# W&B run を起動（トレーニングメトリクスを記録）
train_run = wandb.init(
    project=PROJECT,
    entity=ENTITY,
    job_type="art-training",
    name=f"qwen-1.5b-tool-use-rl",
    config={
        "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
        "num_iterations": NUM_ITERATIONS,
        "rollouts_per_iteration": ROLLOUTS_PER_IT,
        "learning_rate": LEARNING_RATE,
        "task": "arithmetic_tool_use",
    },
)
print(f"W&B run: {train_run.url}")


In [ ]:
# モデルとバックエンドの初期化
backend = LocalBackend(in_process=True)
model = art.TrainableModel(
    name="qwen-1.5b-tool-use",
    project="weave-course-art",
    base_model="Qwen/Qwen2.5-1.5B-Instruct",
    _internal_config=_V100_CONFIG,
)
await model.register(backend)

start_step = await model.get_step()
print(f"学習開始ステップ: {start_step}")

In [ ]:
import os
if os.environ.get("SKIP_EXPENSIVE"):
    print("SKIP_EXPENSIVE=1: このセルをスキップします")
else:
    import random

    for iteration in range(start_step, NUM_ITERATIONS):
        print(f"\n{'='*50}")
        print(f"Iteration {iteration + 1}/{NUM_ITERATIONS}")

        # ── Rollout フェーズ ────────────────────────────────────────────
        problems_batch = random.choices(PROBLEMS, k=ROLLOUTS_PER_IT)
        rollout_results = await art.gather_trajectories(
            [rollout(model, q, a) for q, a in problems_batch]
        )

        # 軌跡を抽出
        trajectories = [r["trajectory"] for r in rollout_results]
        rewards = [r["reward"] for r in rollout_results]
        avg_reward = sum(rewards) / len(rewards)

        print(f"  平均報酬: {avg_reward:.3f} | 成功: {sum(1 for r in rewards if r > 0)}/{len(rewards)}")

        # Weave に軌跡をログ（サンプル表示）
        for r in rollout_results[:2]:
            print(f"  Q: {r['question'][:40]}...")
            print(f"  A: {r['output'][:40]}... (expected: {r['expected']}, reward: {r['reward']})")

        # ── Training フェーズ ───────────────────────────────────────────
        trajectory_groups = [art.TrajectoryGroup(trajectories)]
        await model.delete_checkpoints()
        train_result = await backend.train(
            model,
            trajectory_groups,
            learning_rate=LEARNING_RATE,
        )

        # ── ログ ────────────────────────────────────────────────────────
        metrics = {
            "iteration": iteration + 1,
            "avg_reward": avg_reward,
            "success_rate": sum(1 for r in rewards if r > 0) / len(rewards),
            **(train_result.metrics or {}),
        }
        await model.log(trajectory_groups, metrics=metrics, step=train_result.step, split="train")
        train_run.log(metrics, step=iteration + 1)

    print("\n✅ トレーニング完了!")
    print(f"W&B: {train_run.url}")

## 4. 学習済み LoRA を W&B Artifact として保存

In [ ]:
import shutil

# チェックポイントのパスを取得
checkpoint_dir = Path(".art") / "weave-course-art" / "qwen-1.5b-tool-use"
print("チェックポイントディレクトリ:", checkpoint_dir)
if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.iterdir())
    print("利用可能なチェックポイント:")
    for cp in checkpoints:
        print(f"  {cp}")

# 最新チェックポイントを Artifact として保存
if checkpoint_dir.exists() and list(checkpoint_dir.iterdir()):
    latest_cp = sorted(checkpoint_dir.iterdir())[-1]

    lora_artifact = wandb.Artifact(
        name="qwen-1.5b-tool-use-lora",
        type="model",
        description="Qwen2.5-1.5B-Instruct LoRA trained on arithmetic tool use via ART/CISPO",
        metadata={
            "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
            "training_iterations": NUM_ITERATIONS,
            "final_avg_reward": avg_reward,
            "task": "arithmetic_tool_use",
            "algorithm": "CISPO (GRPO variant)",
        },
    )
    lora_artifact.add_dir(str(latest_cp), name="lora_weights")
    train_run.log_artifact(lora_artifact)
    print(f"LoRA Artifact 保存完了: qwen-1.5b-tool-use-lora")

train_run.finish()
print(f"W&B 実験管理: https://wandb.ai/{ENTITY}/{PROJECT}/runs")


## 5. 学習前後の比較評価

In [ ]:
weave_eval_client = weave.init(f"{ENTITY}/weave-course-art")

class ARTModel(weave.Model):
    art_model_name: str
    _model: art.TrainableModel = None

    def model_post_init(self, __context):
        # 学習済みモデルを再利用
        self._model = model

    @weave.op()
    async def predict(self, question: str) -> dict:
        result = await rollout(self._model, question, expected_answer=-1, max_turns=4)
        return {"output": result["output"]}

@weave.op()
def check_answer(expected: int, output: dict) -> dict:
    numbers = re.findall(r"-?\d+(?:\.\d+)?", output.get("output", "").replace(",", ""))
    predicted = float(numbers[-1]) if numbers else None
    return {"correct": predicted is not None and abs(predicted - expected) < 0.5}

eval_dataset = weave.Dataset(
    name="arithmetic_eval",
    rows=[{"question": q, "expected": a} for q, a in PROBLEMS],
)
weave.publish(eval_dataset)

trained_model = ARTModel(art_model_name="qwen-1.5b-tool-use")
evaluation = weave.Evaluation(
    name="arithmetic_after_training",
    dataset=eval_dataset,
    scorers=[check_answer],
)
summary = await evaluation.evaluate(trained_model)
print("評価結果:", summary)
print(f"Weave UI: https://wandb.ai/{ENTITY}/weave-course-art/weave")


## まとめ

このノートブックでは以下を実装しました：

1. **OpenPipe ART** でツール利用算数エージェントを CISPO（GRPO 変種）で強化学習
2. **Weave** で各ロールアウトの軌跡をトレース（`@weave.op` on rollout）
3. **W&B** でトレーニングメトリクス（報酬推移、成功率）を可視化
4. **W&B Models** に学習済み LoRA Artifact を登録
5. **Weave Evaluation** で学習後の性能を系統的に評価

### 発展的なトピック
- タスクをより複雑に（マルチステップ推論、実際の外部 API 呼び出しなど）
- W&B Serverless バックエンドで大型モデルを学習
- 学習済みモデルを Weave Model として publish して本番環境に統合
